In [3]:
import pandas as pd
import re
from collections import defaultdict

# Lista expandida de gatilhos metacognitivos com padrões mais flexíveis
gatilhos_metacognitivos = {
    'reconhecimento_erro': [
        # Percepção de erro
        "di cuenta", "me equivo", "confund", "cost[oóa]", "dificil",
        "no entend", "no comprend", "no sup", "duda", "error", "mal",
        "fall[oóeéa]", "mistake", "equivoc", "problema", "confus",
        
        # Reflexão sobre erro
        "pens[eéa]", "cre[iyí]", "parec[eéi]", "sent[ií]",
        
        # Dificuldade
        "complicado", "dificultad", "problema", "no puedo", "no pude",
        "no logr[eéo]", "no alcanz[eéo]", "no consig[oua]"
    ],
    'reflexao': [
        # Pensamento reflexivo
        "reflex", "pens[eéa]", "anal", "pregunt", "consider",
        "evalu", "revis", "comprend", "record", "entend",
        "observ", "not[eéa]", "mir[eéa]", "estudi[eéa]",
        
        # Expressões de análise
        "me parec[eéi]", "creo que", "pienso que", "supongo que",
        "me pregunto", "me cuestiono", "debo", "debería",
        
        # Tomada de consciência
        "me doy cuenta", "ahora veo", "ahora entiendo",
        "me percato", "caigo en", "noto que"
    ],
    'monitoramento': [
        # Ações de verificação
        "revis", "verific", "comprob", "repas", "volv[ií]", 
        "consult", "fij[eéa]", "control", "cheque", "mir[eéa]",
        
        # Expressões de monitoramento
        "estoy segur", "quiero ver", "necesito ver",
        "debo revisar", "tengo que ver", "voy a ver",
        
        # Busca de informação
        "busqu[eéa]", "investig", "pregunt", "le[ií]", "mir[eéa]",
        "encuentr", "hall[eéa]"
    ],
    'descoberta': [
        # Momento de descoberta
        "descubr", "sorprend", "llam[oóa] la atenci[oóa]n",
        "entend[ií]", "comprend[ií]", "me di cuenta",
        "not[eéa]", "observ[eéa]", "apareci[oóa]",
        
        # Expressões de surpresa
        "wow", "increible", "interesante", "curioso",
        "fascinante", "impresionante", "sorprendente",
        
        # Mudança de perspectiva
        "cambi[oóa]", "diferente", "otro punto", "nueva forma"
    ],
    'planejamento': [
        # Ações de planejamento
        "planific", "organiz", "decid", "cambi[eéa]", "opt[eéa]",
        "prefer", "eleg", "propu", "prepar", "establec",
        
        # Expressões de intenção
        "voy a", "tengo que", "debo", "necesito", "quiero",
        "planeo", "pienso", "pretendo",
        
        # Estratégias
        "estrategia", "manera", "forma", "método", "paso",
        "primero", "después", "luego", "finalmente"
    ]
}

def encontrar_gatilhos(texto):
    """
    Encontra gatilhos metacognitivos no texto de forma mais flexível.
    """
    if pd.isna(texto):
        return {'gatilhos': [], 'categorias': []}
    
    texto = texto.lower()
    gatilhos_encontrados = set()
    categorias = set()
    
    # Procurar por cada gatilho no texto
    for categoria, padroes in gatilhos_metacognitivos.items():
        for padrao in padroes:
            # Usando regex de forma mais flexível
            if re.search(padrao, texto, re.IGNORECASE):
                gatilhos_encontrados.add(padrao)
                categorias.add(categoria)
    
    return {
        'gatilhos': list(gatilhos_encontrados),
        'categorias': list(categorias)
    }

# Carregar e processar os dados
print("Carregando dados...")
df = pd.read_csv('data/temp/chats_edu.csv', encoding='utf-8', sep=',')

# Aplicar a função de busca de gatilhos
print("Processando mensagens...")
resultados = df['student_query'].apply(encontrar_gatilhos)
df['gatilhos_encontrados'] = resultados.apply(lambda x: x['gatilhos'])
df['categorias_encontradas'] = resultados.apply(lambda x: x['categorias'])

# Deixar apenas as colunas relevantes
df = df[['id','message_id','session_id','student_query', 'timestamp', 'gatilhos_encontrados', 'categorias_encontradas', 'NF', 'ESTADO', 'SEXO']]

# Salvar o DataFrame com os resultados
df.to_csv('data/temp/chats_edu_gatilhos.csv', index=False, encoding='utf-8')


# Filtrar mensagens com gatilhos
df_com_gatilhos = df[df['gatilhos_encontrados'].apply(len) > 0]

# Análise dos resultados
print(f"\nEstatísticas:")
print(f"Total de mensagens analisadas: {len(df)}")
print(f"Mensagens com gatilhos metacognitivos: {len(df_com_gatilhos)}")

# Contagem por categoria
print("\nDistribuição por categoria:")
categoria_counts = defaultdict(int)
for categorias in df_com_gatilhos['categorias_encontradas']:
    for categoria in categorias:
        categoria_counts[categoria] += 1

for categoria, count in sorted(categoria_counts.items(), key=lambda x: x[1], reverse=True):
    percentual = (count / len(df)) * 100
    print(f"{categoria}: {count} mensagens ({percentual:.1f}%)")

# Mostrar exemplos de mensagens com gatilhos
print("\nExemplos de mensagens com gatilhos (primeiros 10):")
for idx, row in df_com_gatilhos.head(10).iterrows():
    print(f"\nMensagem: {row['student_query']}")
    print(f"Gatilhos encontrados: {row['gatilhos_encontrados']}")
    print(f"Categorias: {row['categorias_encontradas']}")

# Análise dos gatilhos mais frequentes
print("\nGatilhos mais frequentes:")
contagem_gatilhos = defaultdict(int)
for gatilhos in df_com_gatilhos['gatilhos_encontrados']:
    for gatilho in gatilhos:
        contagem_gatilhos[gatilho] += 1

print("\nTop 20 gatilhos mais frequentes:")
for gatilho, count in sorted(contagem_gatilhos.items(), key=lambda x: x[1], reverse=True)[:20]:
    percentual = (count / len(df)) * 100
    print(f"{gatilho}: {count} ocorrências ({percentual:.1f}%)")






Carregando dados...
Processando mensagens...

Estatísticas:
Total de mensagens analisadas: 5248
Mensagens com gatilhos metacognitivos: 1511

Distribuição por categoria:
planejamento: 807 mensagens (15.4%)
reflexao: 490 mensagens (9.3%)
reconhecimento_erro: 359 mensagens (6.8%)
monitoramento: 317 mensagens (6.0%)
descoberta: 108 mensagens (2.1%)

Exemplos de mensagens com gatilhos (primeiros 10):

Mensagem: hola tengo problemas con la descarga de autocad
Gatilhos encontrados: ['problema']
Categorias: ['reconhecimento_erro']

Mensagem: 6.	Identificar los requisitos a controlar en la armadura de fundaciones de hormigón armado: Indicar los requisitos a controlar en la armadura de fundaciones de hormigón armado para una vivienda con albañilería reforzada confinada con pilares y vigas de hormigón armado.
Gatilhos encontrados: ['control']
Categorias: ['monitoramento']

Mensagem: normalmente cual el diametro inicial de una red de GLP con cilindros de 45k
Gatilhos encontrados: ['mal']
Categoria

In [4]:
import pandas as pd

df = pd.read_csv('data/metacognition.csv', encoding='utf-8', sep=',')

# Exibir as primeiras linhas do DataFrame
df.head(50)


,id,message_id,session_id,student_query,timestamp,gatilhos_encontrados,categorias_encontradas,novos_gatilhos,novas_categorias,revisar_manual,motivo_revisao
0,2123,39015828-23fc-41f9-9b80-4804035101d4,00019af4-d13d-43be-90aa-981a07434719,cómo instalar firewall mediante dns en cent os 7?,2025-04-28 03:01:00.405998417,[],[],[],[],False,NaN
1,2387,b0643622-feb3-41ef-aa1e-6d77471dc520,00019af4-d13d-43be-90aa-981a07434719,configurar servicio ftp en permanent,2025-04-28 03:57:53.775002992,[],[],[],[],False,NaN
2,2022,076f14fe-f994-4835-a29e-c635fd5a8c10,00019af4-d13d-43be-90aa-981a07434719,como configurar puerto 20 y 21 del servicio ft...,2025-04-28 04:13:47.464000069,[],[],[],[],False,NaN
3,4995,4bcd685c-96eb-4433-83a3-a7f479b3d226,00156c1b-a59a-458a-bb6f-13164383e367,hola tengo problemas con la descarga de autocad,2025-03-27 00:55:01.573997016,['problema'],['reconhecimento_erro'],[],[],True,Classificação metacognitiva existente pode est...
4,7111,17974476-286c-44ed-8ebb-5ca42396057a,00265bef-12aa-492d-a39a-7cf874aed199,Explicame para qué te puedo utilizar,2025-03-27 18:38:42.183997716,[],[],[],[],False,NaN
5,2164,4adf3c05-b9f6-442a-8762-55ea9c30ccbf,002e73d0-86c5-4cb6-a2bb-11318f09ac8e,ISS,2025-05-07 19:31:41.224002101,[],[],[],[],False,NaN
6,3409,50b0e746-c1cd-4ac1-9bc4-57dbe42dced9,00a7e28f-f085-4648-be99-ccd67dd29404,6.\tIdentificar los requisitos a controlar en ...,2025-05-16 13:45:30.946996739,['control'],['monitoramento'],[],[],True,Classificação metacognitiva existente pode est...
7,3495,7bea04f0-b133-429c-96b5-c8fcc6391a90,00a7e28f-f085-4648-be99-ccd67dd29404,7.\tEnumerar las partidas de obra gruesa relac...,2025-05-16 13:46:00.272003640,[],[],[],[],False,NaN
8,3765,f1b25c54-4067-4593-8514-93a2d02ac89a,00a7e28f-f085-4648-be99-ccd67dd29404,8.\tDescribir las partidas de obra gruesa rela...,2025-05-16 13:48:42.535998538,[],[],[],[],False,NaN
9,338,524a2897-023b-43b3-8181-6e5b8e45177d,00c3806d-8df9-4539-bda9-7a39e9ffafb2,que es la geologia,2025-05-06 20:37:48.821997967,[],[],[],[],False,NaN
